# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

In [1]:
# ⏱ Додано для питання 22: вимірювання часу виконання кожної клітинки.
# Після кожної клітинки друкується її тривалість; підсумок виводить функція timing_report() у розділі з відповідями.
import time
from IPython import get_ipython

NOTEBOOK_START = time.time()
CELL_TIMES = []  # (перший рядок клітинки, тривалість у секундах)
_timer = {"t0": None}


def _pre_run_cell(info):
    _timer["t0"] = time.time()


def _post_run_cell(result):
    if _timer["t0"] is None:
        return
    seconds = time.time() - _timer["t0"]
    _timer["t0"] = None
    lines = (getattr(result.info, "raw_cell", "") or "").strip().splitlines()
    CELL_TIMES.append((lines[0][:70] if lines else "", seconds))
    print(f"⏱ {seconds:.1f} с")


ip = get_ipython()
for _event, _callback in globals().get("_timing_callbacks", []):  # повторний запуск не дублює обробники
    try:
        ip.events.unregister(_event, _callback)
    except ValueError:
        pass
_timing_callbacks = [("pre_run_cell", _pre_run_cell), ("post_run_cell", _post_run_cell)]
for _event, _callback in _timing_callbacks:
    ip.events.register(_event, _callback)


def timing_report(top=6):
    """Друкує сумарний час усіх клітинок і повертає таблицю з найдовшими."""
    import pandas as pd

    df = pd.DataFrame(CELL_TIMES, columns=["Клітинка (перший рядок)", "Час, с"])
    total = df["Час, с"].sum()
    print(f"Клітинок виконано: {len(df)}")
    print(f"Сумарний час виконання клітинок: {total / 60:.1f} хв ({total:.0f} с)")
    print(f"Від запуску ноутбука минуло: {(time.time() - NOTEBOOK_START) / 60:.1f} хв (разом із паузами між клітинками)")
    return df.sort_values("Час, с", ascending=False).head(top).round(1)

In [2]:
# trl — бібліотека для донавчання мовних моделей (SFT, DPO тощо),
# peft — реалізація LoRA та інших методів PEFT, bitsandbytes — 4-бітна квантизація для QLoRA
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
# Colab має застарілу бібліотеку torchao, з якою peft не імпортується; для цього демо вона не потрібна
!pip uninstall -q -y torchao

⏱ 18.2 с


In [3]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Памʼять GPU: 15.6 GB
⏱ 59.2 с


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [4]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}
⏱ 8.0 с


In [5]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']
⏱ 1.8 с


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [6]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371
⏱ 1.0 с


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [7]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

⏱ 0.0 с


### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) — на ньому модель вчиться.
- **Валідаційний** (200) — на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) — на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 — на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [8]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'content': 'Classify the customer message. Reply with JSON only.',
  'role': 'system'},
 {'content': 'I have got to locate hte bills from {{Person Name}}',
  'role': 'user'},
 {'content': '{"category": "INVOICE", "intent": "check_invoice"}',
  'role': 'assistant'}]

⏱ 1.2 с


## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [9]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

⏱ 6.5 с


### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [10]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів
⏱ 0.0 с


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [11]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

⏱ 0.0 с


## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [12]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

⏱ 0.0 с


Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` — loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` — кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування — 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` — для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` — кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` — змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` — активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше памʼяті.

In [13]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,                  # діалог із цього датасету після шаблону займає близько 100 токенів
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # ефективний розмір батча: 4 × 2 = 8 прикладів
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову памʼять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # Для QLoRA бібліотека TRL переводить ваги адаптера в bf16, який T4 не підтримує.
    # Повертаємо їх у fp32: змішана точність fp16 потребує, щоб параметри, що тренуються, були у fp32
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова памʼять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

⏱ 0.0 с


## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити памʼять для наступної.

In [14]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

⏱ 0.0 с


In [15]:
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB

До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (8 outputs left out)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.506702,0.400654,0.385706,0.912863,4323.000000
20,0.262003,0.117909,0.144082,0.963722,8672.000000
30,0.067879,0.042200,0.062891,0.988512,13032.000000
40,0.039489,0.023353,0.033976,0.991258,17377.000000
50,0.018554,0.009793,0.022544,0.997490,21714.000000
60,0.006635,0.006441,0.016143,0.998490,26070.000000
70,0.011565,0.005416,0.014622,0.998751,30452.000000
80,0.006362,0.004750,0.012457,0.999248,34813.000000
90,0.004502,0.005560,0.012551,0.999255,39116.000000
100,0.014816,0.005548,0.012468,0.999255,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,98.00
"Тривалість, хв",NaN,2.62
"Пікова памʼять GPU, GB",NaN,1.42
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


⏱ 297.0 с


In [16]:
run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (3 outputs left out)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,2.098375,0.381385,0.314538,0.916777,4323.000000
20,0.270690,0.115679,0.170246,0.964285,8672.000000
30,0.069361,0.040015,0.043759,0.987005,13032.000000
40,0.036743,0.019349,0.028457,0.992774,17377.000000
50,0.019136,0.018527,0.025466,0.993261,21714.000000
60,0.010345,0.009014,0.014346,0.997251,26070.000000
70,0.010701,0.007474,0.010449,0.997504,30452.000000
80,0.009415,0.006033,0.009285,0.998250,34813.000000
90,0.013191,0.005254,0.009085,0.998754,39116.000000
100,0.008208,0.005001,0.008974,0.998500,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",62.7,99.30
"Точність intent, %",57.4,96.90
"Тривалість, хв",NaN,2.50
"Пікова памʼять GPU, GB",NaN,3.83
Validation loss,NaN,0.01
"Адаптер, MB",NaN,69.78


⏱ 394.7 с


In [17]:
run_lora("Qwen/Qwen3-4B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (3 outputs left out)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.921634,0.262582,0.258228,0.933482,4323.000000
20,0.178605,0.087500,0.102374,0.971230,8672.000000
30,0.058869,0.020346,0.034087,0.994246,13032.000000
40,0.024370,0.013872,0.019476,0.996760,17377.000000
50,0.011457,0.007288,0.015018,0.997488,21714.000000
60,0.005324,0.003794,0.008977,0.998740,26070.000000
70,0.008160,0.004124,0.007033,0.998490,30452.000000
80,0.006622,0.003857,0.006025,0.998240,34813.000000
90,0.009639,0.003062,0.006146,0.998995,39116.000000
100,0.008364,0.002979,0.006154,0.998995,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",88.1,100.00
"Точність intent, %",72.5,97.60
"Тривалість, хв",NaN,3.73
"Пікова памʼять GPU, GB",NaN,8.75
Validation loss,NaN,0.00
"Адаптер, MB",NaN,132.19


⏱ 835.9 с


### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [18]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00
"Точність intent, %",25.6,98.00,57.4,96.90,72.5,97.60
"Тривалість, хв",NaN,2.62,NaN,2.50,NaN,3.73
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75
Validation loss,NaN,0.01,NaN,0.01,NaN,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19


⏱ 0.0 с


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` — тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` — для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [19]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.737474,0.605169,0.271417,0.882885,4323.000000
20,0.282767,0.056560,0.095230,0.985241,8672.000000
30,0.037163,0.049979,0.014630,0.987977,13032.000000
40,0.038285,0.020181,0.016560,0.993763,17377.000000
50,0.012865,0.011908,0.009553,0.996995,21714.000000
60,0.003235,0.009957,0.007164,0.997997,26070.000000
70,0.006116,0.009358,0.008030,0.997501,30452.000000
80,0.011116,0.012613,0.006138,0.996752,34813.000000
90,0.004606,0.011770,0.004709,0.997001,39116.000000
100,0.021278,0.010938,0.004558,0.997001,43449.000000


📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,Qwen3-0.6B LoRA,Qwen3-0.6B повне
"Чистий JSON, %",100.00,100.00
"Валідний JSON, %",100.00,100.00
"Точність category, %",99.70,99.40
"Точність intent, %",98.00,95.90
"Тривалість, хв",2.62,1.69
"Пікова памʼять GPU, GB",1.42,10.80
Validation loss,0.01,0.01
"Адаптер, MB",40.42,NaN


⏱ 151.1 с


## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [20]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

README.md                                0.0 MB
adapter_config.json                      0.0 MB
adapter_model.safetensors              132.2 MB
chat_template.jinja                      0.0 MB
tokenizer.json                          11.4 MB
tokenizer_config.json                    0.0 MB
training_args.bin                        0.0 MB

Для порівняння, ваги Qwen3-4B у fp16: ~8 GB
⏱ 0.0 с


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [21]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)

# Злиту модель можна зберегти як звичайну: model.save_pretrained("qwen3-4b-merged")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM
⏱ 32.6 с


Перевіримо злиту модель на власних повідомленнях:

In [22]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_period"}

📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "order_confirmed"}

📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}

📩 I guess, you have shipped to a wrong address
🤖 {"category": "SHIPPING", "intent": "shipping_address"}

⏱ 4.0 с


In [23]:
del model
free_memory()

⏱ 0.5 с


## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб — найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** — 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** — 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 — формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше — у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [24]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-бітний формат, розрахований на нормально розподілені ваги
    bnb_4bit_compute_dtype=torch.float16,   # обчислення виконуються у fp16
    bnb_4bit_use_double_quant=True,         # квантизуються також масштаби квантизації: ще трохи менше памʼяті
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Памʼять GPU після завантаження: 2.7 GB
⏱ 29.7 с


In [25]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.881794,0.281242,0.299897,0.930995,4323.000000
20,0.190578,0.107701,0.101937,0.963978,8672.000000
30,0.068624,0.027191,0.043979,0.992731,13032.000000
40,0.029281,0.013553,0.021350,0.996752,17377.000000
50,0.012995,0.011981,0.017030,0.996732,21714.000000
60,0.009341,0.005340,0.010753,0.998740,26070.000000
70,0.007281,0.006421,0.008959,0.997990,30452.000000
80,0.003197,0.006316,0.007569,0.997990,34813.000000
90,0.012025,0.005219,0.007346,0.997990,39116.000000
100,0.009433,0.004933,0.007357,0.997990,43449.000000


{'Тривалість, хв': 5.075132811069489,
 'Пікова памʼять GPU, GB': 3.664846336,
 'Validation loss': 0.004933448042720556}

⏱ 448.8 с


## 9. Підсумкове порівняння

In [26]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA,Qwen3-0.6B повне,Qwen3-4B QLoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00,99.40,99.90
"Точність intent, %",25.6,98.00,57.4,96.90,72.5,97.60,95.90,96.60
"Тривалість, хв",NaN,2.62,NaN,2.50,NaN,3.73,1.69,5.08
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75,10.80,3.66
Validation loss,NaN,0.01,NaN,0.01,NaN,0.00,0.01,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19,NaN,NaN


⏱ 0.0 с


**Що варто помітити:**
- LoRA тренує близько 1–2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й памʼяті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і памʼяттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після — лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше памʼяті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

---
# Відповіді на теоретичні питання (Лабораторна 5, п. 0.1, питання 22–30)

Клітинки з кодом у цьому розділі самі збирають дані з вашого запуску. Усі числа в текстових відповідях узято з виводів цього запуску (Google Colab, GPU Tesla T4).

## Питання 22. Як довго виконувалися клітинки? Чи були проблеми з запуском?

In [27]:
# Тривалість виконання клітинок ноутбука (дані збирає обробник із першої клітинки)
print("Пристрій:", torch.cuda.get_device_name())
timing_report()

Пристрій: Tesla T4
Клітинок виконано: 25
Сумарний час виконання клітинок: 38.2 хв (2290 с)
Від запуску ноутбука минуло: 38.2 хв (разом із паузами між клітинками)


,Клітинка (перший рядок),"Час, с"
15,"run_lora(""Qwen/Qwen3-4B"")",835.9
23,"trainer, qlora_stats = train(model, ""Qwen3-4B-...",448.8
14,"run_lora(""Qwen/Qwen3-1.7B"")",394.7
13,"run_lora(""Qwen/Qwen3-0.6B"")",297.0
17,"model = AutoModelForCausalLM.from_pretrained(""...",151.1
1,import gc,59.2


⏱ 0.0 с


**Відповідь.** Усі клітинки виконалися за **38,2 хв (2290 с)** на **Tesla T4**. Найдовші з них:
- `run_lora("Qwen/Qwen3-4B")`: **835,9 с** (≈14 хв);
- тренування QLoRA для 4B: 448,8 с (≈7,5 хв);
- `run_lora("Qwen/Qwen3-1.7B")`: 394,7 с;
- `run_lora("Qwen/Qwen3-0.6B")`: 297,0 с;
- повне донавчання 0,6B: 151,1 с;
- перша клітинка з імпортами: 59,2 с.

Ноутбук довгий, бо в ньому тричі виконується LoRA (0,6B / 1,7B / 4B, кожного разу з оцінкою на 1000 прикладів до й після), одне повне донавчання та QLoRA, яка повільніша через деквантизацію ваг.

**Проблеми з запуском:** помилок у виводі немає, а `CUDA out of memory` не траплялась. З'являлися лише попередження:
- про неавтентифіковані запити до Hugging Face Hub (`HF_TOKEN`), не критично;
- `UserWarning` від PEFT: *«The PEFT config's `base_model_name_or_path` was renamed…»*. Воно з'являлось у `run_lora` для 1,7B і 4B, імовірно тому, що той самий об'єкт `peft_config` використовується для різних моделей і PEFT оновлює в ньому назву базової моделі. На результат навчання це не впливає.

## Питання 23. Прокоментуйте вивід «Памʼять GPU після завантаження». Чи очікуваний результат?

Памʼять ваг ≈ **кількість параметрів × байтів на параметр**. Кількість параметрів надруковано в тих самих рядках виводу (`run_lora`: «0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB» тощо).

**Відповідь.** Так, результат **очікуваний**. У fp16 кожен параметр займає 2 байти, тому очікувана памʼять ваг дорівнює кількості параметрів × 2 байти. Порівняння з моїм виводом:

| Модель | Параметрів (з виводу) | Очікувано в fp16 (×2 байти) | Виміряно після завантаження |
|---|---|---|---|
| Qwen3-0.6B | 0,60 млрд | 1,2 GB | 1,2 GB |
| Qwen3-1.7B | 1,72 млрд | 3,4 GB | 3,5 GB |
| Qwen3-4B | 4,02 млрд | 8,0 GB | 8,1 GB |
| Qwen3-4B, QLoRA (NF4) | 4,02 млрд | ≈2,6 GB (груба оцінка\*) | 2,7 GB |

\* У NF4 ваги лінійних шарів займають ≈0,5 байта на параметр, а матриця вбудовувань (151 936 × 2560 ≈ 0,39 млрд параметрів) лишається у fp16: (4,02 − 0,39) × 0,5 + 0,39 × 2 ≈ 2,6 GB.

Виміряне й очікуване збігаються з точністю ≈0,1 GB. QLoRA стискає ті самі ваги втричі (8,1 → 2,7 GB), а не вчетверо, бо вбудовування лишаються у fp16.

Важливі застереження:
- **Це памʼять лише під ваги.** Пікова памʼять під час тренування вища: для LoRA 1,42 / 3,83 / 8,75 GB (0,6B / 1,7B / 4B), тобто на 0,2 / 0,3 / 0,65 GB більше за памʼять після завантаження (активації, градієнти й стан оптимізатора для адаптера, який становить лише ≈1% параметрів). Для QLoRA 4B пік 3,66 GB (≈+1 GB).
- `torch.cuda.memory_allocated()` показує лише тензори PyTorch. `nvidia-smi` покаже трохи більше (контекст CUDA, кеш аллокатора).
- Саме тому 4B у fp16 для LoRA вміщується в T4 (15,6 GB), а повне донавчання ні. У моєму запуску повне донавчання навіть 0,6B вимагало **10,8 GB** (≈0,6 млрд × 16 байтів ≈ 9,6 GB плюс активації), тож для 4B було б ≈64 GB.

## Питання 24. Що робить `with torch.no_grad()` у функції `classify()`?

**Відповідь.** Блок вимикає **відстеження градієнтів**: PyTorch не будує обчислювальний граф і не зберігає проміжні активації, потрібні для зворотного проходу. На інференсі градієнти не потрібні, бо ми нічого не навчаємо, тому це **економить памʼять і трохи пришвидшує обчислення**. Без цього блока памʼять зростала б на кожен виклик, що особливо відчутно для батчів і великих моделей на 15 GB T4. Варто зауважити, що в сучасних версіях `transformers` метод `generate()` сам працює без градієнтів, тож тут `no_grad` — явна страховка, яку все одно прийнято писати. Це не те саме, що `model.eval()`, який окремо вимикає dropout (і dropout в адаптері LoRA).

## Питання 25. Для чого у `classify()` задано `do_sample=False`?

**Відповідь.** `do_sample=False` вмикає **жадібне декодування** (greedy): на кожному кроці береться токен із найбільшою ймовірністю, без випадковості. Для нашої задачі це потрібно, бо:
- відповідь має бути **детермінованою**: те саме звернення завжди отримує ту саму мітку, інакше маршрутизація звернень була б «лотереєю»;
- JSON треба видати в **строго заданому форматі**; випадковий вибір менш імовірного токена може зламати синтаксис чи підставити сусідню мітку;
- метрики на тестовому наборі стають **відтворюваними**, і порівняння «до/після» чесне: різниця не пояснюється випадковістю генерації.

Семплінг (`do_sample=True`, `temperature`) доречний там, де потрібна різноманітність, наприклад у творчих текстах чи чаті. Для коротких структурованих відповідей він лише шкодить.

## Питання 26. В демо 1 класифікатор отримано заміною голови. Чому не застосувати той самий підхід у демо 3?

**Відповідь.** Принципово це **можливо** (для Qwen3 існує `AutoModelForSequenceClassification`, яка бере вектор останнього токена й додає лінійну голову), але в задачі демо 3 такий підхід **не підходить або поступається SFT**:
1. **Різна архітектура.** BERT — двонапрямлений енкодер зі спеціальним токеном `[CLS]`, який ідеально підходить для голови класифікації. Qwen3 — авторегресійний декодер (causal LM): токена `[CLS]` немає, а його «голова» — це LM head, що передбачає **наступний токен зі словника (≈152 тис. токенів)**. Щоб класифікувати, довелося б вигадувати агрегацію, наприклад брати останній токен.
2. **Формат результату.** Нам потрібен **структурований текст** (`{"category": …, "intent": …}`) з двома полями. Для голови потрібно було б дві окремі голови (11 і 27 класів) і окремий код розбору, а SFT дає потрібний формат безпосередньо.
3. **Нова голова навчається з нуля**, тому потребує більше даних і не використовує знання, які вже є в моделі про слова на кшталт `cancel_order` чи `refund`. SFT дає змогу модель опиратися на семантику назв міток.
4. **Гнучкість.** Додати нову мітку з головою означає змінити розмір голови й перенавчити її, а з SFT це лише нові приклади. Модель при цьому зберігає здатність до звичайного діалогу.
5. **Вартість.** Для чистої класифікації з великою кількістю даних енкодер із головою (≈110M параметрів) значно дешевший і швидший за декодер на 4B, який ще й генерує токени. Тож вибір залежить від задачі: енкодер із головою — для чистої класифікації з фіксованими класами, SFT — коли потрібні формат, гнучкість і семантика.

## Питання 27. Токени короткого й детального промпту та вартість 10 000 звернень на день

In [29]:
# Питання 27: кількість токенів у промптах і оцінка вартості обробки 10 000 звернень на день
import numpy as np

n_short = len(tokenizer(SHORT_PROMPT)["input_ids"])
n_long = len(tokenizer(LONG_PROMPT)["input_ids"])
print(f"Лише системний промпт: короткий — {n_short} токенів, детальний — {n_long} токенів (різниця {n_long - n_short}).")


def request_tokens(system_prompt, message):
    """Токенів у всьому запиті: шаблон розмови + системний промпт + звернення клієнта."""
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    return len(tokenizer(text)["input_ids"])


sample = [ex["messages"][1]["content"] for ex in test_dataset.select(range(200))]
avg_short = np.mean([request_tokens(SHORT_PROMPT, m) for m in sample])
avg_long = np.mean([request_tokens(LONG_PROMPT, m) for m in sample])

REQUESTS_PER_DAY = 10_000
PRICE_PER_1M_INPUT = 0.20  # USD за 1 млн вхідних токенів: УМОВНЕ значення для розрахунку, підставте ціну свого провайдера
rows = []
for name, avg in [("Детальний промпт (до донавчання)", avg_long), ("Короткий промпт (після донавчання)", avg_short)]:
    daily = avg * REQUESTS_PER_DAY
    rows.append({
        "варіант": name,
        "токенів у запиті (середнє)": round(avg),
        "вхідних токенів на день": round(daily),
        "умовна вартість, $/день": daily / 1e6 * PRICE_PER_1M_INPUT,
        "умовна вартість, $/місяць": daily * 30 / 1e6 * PRICE_PER_1M_INPUT,
    })
cost = pd.DataFrame(rows).set_index("варіант")
display(cost.round(2))
saved = (avg_long - avg_short) * REQUESTS_PER_DAY
print(f"Економія: {saved:,.0f} вхідних токенів на день ({100 * (1 - avg_short / avg_long):.0f}% від запиту).")

Лише системний промпт: короткий — 11 токенів, детальний — 166 токенів (різниця 155).


,токенів у запиті (середнє),вхідних токенів на день,"умовна вартість, $/день","умовна вартість, $/місяць"
варіант,,,,
Детальний промпт (до донавчання),193,1930300,0.39,11.58
Короткий промпт (після донавчання),38,380300,0.08,2.28


Економія: 1,550,000 вхідних токенів на день (80% від запиту).
⏱ 0.2 с


**Відповідь.** Системний промпт має **11 токенів** (короткий) проти **166** (детальний), різниця 155 токенів. З урахуванням шаблону розмови й звернення клієнта (середнє за 200 прикладами) весь запит займає **193 токени** проти **38**, тобто короткий промпт зменшує запит на **80%**. Детальний промпт містить перелік усіх 11 категорій і 27 намірів, а короткий має лише одне речення. Цю різницю **модель обробляє на кожному запиті заново**, тому за 10 000 звернень на день вийшло **1 930 300** вхідних токенів проти **380 300**, тобто економія ≈**1,55 млн токенів на день**. За умовною ціною $0,20 за 1 млн вхідних токенів це $0,39 проти $0,08 на день (≈$11,6 проти ≈$2,3 на місяць). Суми **умовні** (ціну задано змінною `PRICE_PER_1M_INPUT`), важливе тут співвідношення: вхідна частина дешевшає приблизно в 5 разів. Вихід (JSON ≈ 25 токенів) в обох випадках однаковий.

Токени коштують не лише грошей:
- довший промпт збільшує **затримку** (prefill) і зменшує **пропускну здатність** власного GPU (запитів на секунду);
- він займає частину контекстного вікна й пам'яті KV-кешу;
- кешування префікса (prompt caching), якщо провайдер його підтримує, частково знижує вартість довгого промпту, але донавчання дає економію й без залежності від такої функції.

Донавчання коштує **один раз** (хвилини тренування на T4), а економія на промпті діє **на кожному запиті**, тому на великому обсязі воно швидко окупається.

## Питання 28. Для чого колонка `chat_template_kwargs` з `enable_thinking=False`? Що було б, якби донавчали в режимі `/think`?

**Відповідь.** Qwen3 — гібридна модель із двома режимами. З `enable_thinking=False` шаблон розмови вмикає режим **без міркувань**: до відповіді додається порожній блок `<think></think>`, і модель відразу видає результат. `SFTTrainer` передає значення з колонки в `apply_chat_template`, тому **тренувальні приклади форматуються так само, як промпти на інференсі** (`classify_batch` також викликає шаблон з `enable_thinking=False`). Наші еталонні відповіді — чистий JSON без жодних міркувань, тож такий режим відповідає даним.

Якби донавчали в режимі `/think` на цих самих прикладах:
- **Невідповідність навчання й інференсу.** Модель бачила б один префікс відповіді в навчанні, а в роботі отримувала б інший, і поведінка стала б нестабільною (зайвий текст, порожні чи зайві блоки `<think>`).
- **Модель відучилась би міркувати.** У режимі `/think` вона бачила б приклади, де замість міркувань одразу йде JSON, тож вивчила б правило «навіть у режимі think відповідай без міркувань». Втратилась би здатність, яка є ключовою для режиму `/think` (аналогічне попередження є в кінці демо 2).
- Було б марнування часу: навчання на прикладах із режимом, який ми не використовуємо.

## Питання 29. Що означає `assistant_only_loss=True`? Без нього на яких токенах модель вчилась би?

**Відповідь.** Параметр означає, що **loss рахується лише на токенах відповіді асистента**. Для решти токенів (системний промпт, звернення клієнта, службова розмітка) мітки замінюються на −100, і функція втрат їх ігнорує.

**Без нього** SFT за замовчуванням рахує loss на **всіх токенах** відформатованого діалогу: модель вчилася б ще й передбачати системний промпт і самі звернення клієнтів. Для нашої задачі це погано:
- **розмитий сигнал.** Звернення клієнта довгі й різноманітні, їхні токени важко передбачити, а JSON з двома мітками короткий. Тож більшість градієнта пішла б не на те, що нам потрібно (правильні мітки), а на імітацію чужих текстів;
- **марна ємність.** Модель витрачала б вагу адаптера на запам'ятовування стандартного системного промпту й фраз зі звернень (а це перенавчання на вхідних даних);
- **неінформативний validation loss.** Він би змішував якість міток із передбачуваністю звернень, тож за ним важко відстежувати перенавчання.

Нам потрібно лише одне: за користувацьким текстом видати правильний JSON. Тому loss має бути лише на відповіді асистента.

## Питання 30. Порівняйте LoRA і QLoRA за памʼяттю, часом тренування й якістю. Коли обрати QLoRA?

In [30]:
# Питання 30: LoRA проти QLoRA на Qwen3-4B (дані з розділів 5 і 8)
def pick(d, prefix):
    """Значення зі словника метрик за початком назви ключа (щоб не залежати від апострофів у назвах)."""
    return next(v for k, v in d.items() if k.startswith(prefix))


lora_4b = results["Qwen3-4B"]["LoRA"]
qlora_4b = qlora_metrics | qlora_stats
fields = ["Пікова", "Тривалість", "Validation", "Точність category", "Точність intent"]
labels = ["Пікова пам'ять GPU, GB", "Тривалість, хв", "Validation loss", "Точність category, %", "Точність intent, %"]
comparison = pd.DataFrame({
    "LoRA (fp16)": [pick(lora_4b, f) for f in fields],
    "QLoRA (NF4)": [pick(qlora_4b, f) for f in fields],
}, index=labels)
display(comparison.round(2))

mem_l, mem_q = comparison.loc["Пікова пам'ять GPU, GB"]
t_l, t_q = comparison.loc["Тривалість, хв"]
i_l, i_q = comparison.loc["Точність intent, %"]
print(f"Памʼять: QLoRA потребує на {100 * (1 - mem_q / mem_l):.0f}% менше ({mem_l:.1f} → {mem_q:.1f} GB).")
print(f"Час: QLoRA тренується у {t_q / t_l:.2f} раза {'довше' if t_q > t_l else 'швидше'} ({t_l:.1f} → {t_q:.1f} хв).")
print(f"Якість (intent): LoRA {i_l:.1f}% проти QLoRA {i_q:.1f}% (різниця {i_q - i_l:+.1f} п. п.).")

,LoRA (fp16),QLoRA (NF4)
"Пікова пам'ять GPU, GB",8.75,3.66
"Тривалість, хв",3.73,5.08
Validation loss,0.00,0.00
"Точність category, %",100.00,99.90
"Точність intent, %",97.60,96.60


Памʼять: QLoRA потребує на 58% менше (8.7 → 3.7 GB).
Час: QLoRA тренується у 1.36 раза довше (3.7 → 5.1 хв).
Якість (intent): LoRA 97.6% проти QLoRA 96.6% (різниця -1.0 п. п.).
⏱ 0.0 с


**Відповідь.** Числа для Qwen3-4B наведено в таблиці й підсумкових рядках вище. Загальна картина така:
- **Памʼять:** QLoRA зберігає заморожені ваги у 4 бітах (NF4), тож потребує значно менше памʼяті (ваги ≈2,5–3 GB замість ≈8 GB), хоч пік тренування складається ще з активацій і стану адаптера, тому економія менша за ×3. У моєму запуску пік 8,75 GB проти 3,66 GB, тобто на 58% менше (у ≈2,4 раза).
- **Час:** QLoRA **повільніша**, бо перед кожним множенням 4-бітні ваги потрібно деквантизувати у fp16. Це плата за економію памʼяті. У моєму запуску 5,08 хв проти 3,73 хв (у 1,36 раза довше).
- **Якість:** у статті QLoRA та на практиці результат майже збігається з LoRA. У моєму запуску: category 100,0% (LoRA) проти 99,9% (QLoRA), intent 97,6% проти 96,6%, validation loss близький до нуля в обох. Різниця в 1 п. п. за intent — це 10 прикладів із 1000; стандартна похибка такої оцінки близько 0,5 п. п., тож це на межі шуму, і стверджувати, що QLoRA гірша, не можна. Квантизація справді вносить невелику похибку, але адаптер у fp32 здебільшого її компенсує.

**Коли обрати QLoRA:**
- модель **не вміщується** в GPU у fp16 (наприклад, 7–8B і більше на T4/16 GB або 13B+ на 24 GB), а LoRA дала б `out of memory`;
- обмежені ресурси (домашня чи безкоштовна GPU), потрібно мати більший батч чи довші послідовності в тій же памʼяті;
- швидко перевіряємо ідею на більшій моделі.

**Коли краще звичайна LoRA:** якщо модель і так вміщується (як 0,6–4B на T4): вона швидша, простіша, без квантизаційної похибки, а адаптер можна злити в ваги fp16 без втрат. Тому в цьому демо я використав би LoRA для 4B, а QLoRA залишив би для моделей, які без неї не завантажити.

**Підсумок моїх чисел:** QLoRA заощадила **5,1 GB** пікової памʼяті (58%: 8,75 → 3,66 GB), але тренувалась у **1,36 раза довше** (5,08 проти 3,73 хв); точність intent 96,6% проти 97,6%, category 99,9% проти 100%.